# Trigger Revision  -  MHIBFEWS (Philippines Storms)

This notebook explores a revision of the anticipatory action trigger for the Philippines storms programme using the MHIBFEWS framework. It fetches historical tropical cyclone track data from the IBTrACS database for the Western Pacific basin and loads Philippine administrative boundaries to support spatial analysis. The analysis aims to evaluate and refine trigger thresholds by examining storm track patterns relative to target areas in the Philippines.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import pandas as pd
import geopandas as gpd
import ocha_stratus as stratus
from shapely.geometry import LineString, Polygon, box
from shapely.ops import unary_union
from rasterstats import zonal_stats

In [ ]:
phl_admin_1 = stratus.codab.load_codab_from_fieldmaps(iso3="PHL", admin_level=1)
phl_admin_3 = stratus.codab.load_codab_from_fieldmaps(iso3="PHL", admin_level=3)

In [ ]:
# load world pop raster using stratus
global_pop_raster = stratus.open_blob_cog(blob_name="worldpop/pop_count/global_pop_2026_CN_1km_R2025A_UA_v1.tif", stage='dev', container_name='raster')   

In [ ]:
combined_input_data = stratus.load_csv_from_blob(blob_name="pa-aa-phl-storms/processed/combined_input_data.csv", stage='dev', container_name='projects')
combined_input_data.head()

In [ ]:
phl_cerf_data = stratus.load_csv_from_blob(blob_name="pa-aa-phl-storms/processed/phl_cerf_allocations.csv", stage='dev', container_name='projects')
phl_cerf_data.head()

In [ ]:
def fetch_ibtracs_data(
    start_date: pd.Timestamp, end_date: pd.Timestamp
) -> pd.DataFrame:
    """Fetch IBTRACS data for a date range

    Parameters
    ----------
    start_date
        Start date
    end_date
        End date

    Returns
    -------
    pd.DataFrame
        DataFrame with IMERG data
    """
    query = f"""
    SELECT *
    FROM storms.ibtracs_tracks_geo
    WHERE
        basin = 'WP'
        AND valid_time BETWEEN '{start_date.date()}' AND '{end_date.date()}'
    """
    return gpd.read_postgis(query, stratus.get_engine(stage="prod"), geom_col="geometry")

## Storm Tracks over the Philippines (2024)

Western Pacific tropical cyclone tracks for 2024, colored by maximum recorded wind speed, plotted over Philippine admin level 1 boundaries. Only tracks that pass within or near the Philippine archipelago are shown.

In [ ]:
# Fetch 2024 storm track points
tracks_pts = fetch_ibtracs_data(pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31"))

# Build one LineString per storm with name and max wind speed
def build_track_lines(gdf):
    rows = []
    for storm_id, grp in gdf.sort_values("valid_time").groupby("storm_id"):
        coords = list(zip(grp.geometry.x, grp.geometry.y))
        if len(coords) >= 2:
            peak_idx = grp["wind_speed"].idxmax() if grp["wind_speed"].notna().any() else grp.index[-1]
            rows.append({
                "storm_id": storm_id,
                "name": storm_id.split("_")[0].capitalize(),
                "geometry": LineString(coords),
                "max_wind": grp["wind_speed"].max(),
                "label_lon": grp.loc[peak_idx, "geometry"].x,
                "label_lat": grp.loc[peak_idx, "geometry"].y,
            })
    return gpd.GeoDataFrame(rows, crs="EPSG:4326")

all_tracks = build_track_lines(tracks_pts)

# Keep only tracks that pass through the Philippine region
phl_box = box(114, 4, 130, 22)
nearby_tracks = all_tracks[all_tracks.intersects(phl_box)].copy()

# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 11))
ax.set_facecolor("#cde0f0")

# Admin regions
phl_admin_1.plot(ax=ax, color="#f0ece3", edgecolor="#aaa", linewidth=0.5, zorder=2)

# Color tracks by max wind speed
vmin, vmax = 30, 140
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.plasma

for _, row in nearby_tracks.iterrows():
    wind = row["max_wind"]
    color = cmap(norm(wind)) if pd.notna(wind) else "#999999"
    xs, ys = row.geometry.xy
    ax.plot(xs, ys, color=color, linewidth=2, alpha=0.85, zorder=3)

    # Mark peak intensity point
    ax.scatter(row["label_lon"], row["label_lat"], color=color, s=40,
               edgecolors="white", linewidths=0.6, zorder=5)

    # Storm name label
    ax.annotate(
        row["name"],
        xy=(row["label_lon"], row["label_lat"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8.5,
        fontweight="bold",
        color="white",
        zorder=6,
        bbox=dict(boxstyle="round,pad=0.2", fc=color, ec="none", alpha=0.75),
    )

# Colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=18)
cbar.set_label("Max wind speed (knots)", fontsize=10)
cbar.ax.tick_params(labelsize=9)

ax.set_xlim(112, 135)
ax.set_ylim(4, 22)
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title(
    "Tropical Cyclone Tracks near the Philippines - 2024\nLabeled at peak intensity, colored by maximum wind speed",
    fontsize=12,
    fontweight="bold",
    pad=12,
)
ax.tick_params(labelsize=9)

plt.tight_layout()
plt.show()

## Municipality Exposure Analysis (1980-2024)

We estimate how many Philippine municipalities were exposed to extreme winds for every typhoon on record since 1980, using two approaches:

| | Approach | Wind threshold | Spatial footprint |
|---|---|---|---|
| **Option A** | Fixed buffer | 194 km/h gust equivalent (~80 kt sustained) | 100 km circle around each intense track point |
| **Option B** | IBTrACS wind radii | 64 kt (118 km/h) | Asymmetric polygon from reported quadrant radii |

Option A is robust and consistent across the full record. Option B captures storm asymmetry but depends on data availability  -  coverage of 64-knot radii in IBTrACS is checked before use.

In [ ]:
# ── Shared constants ─────────────────────────────────────────────────────────
GUST_THRESHOLD_KMH  = 194       # target gust wind speed (km/h)
GUST_FACTOR         = 1.3       # standard TC gust factor
BUFFER_M            = 100_000   # Option A: 100 km fixed buffer (metres)
CRS_METRIC          = "EPSG:32651"  # UTM Zone 51N, centred on the Philippines

# ── Fetch full historical record (1980-present) ──────────────────────────────
tracks_hist = fetch_ibtracs_data(pd.Timestamp("1980-01-01"), pd.Timestamp("2024-12-31"))

# ── Project admin-3 once; reused by both options ─────────────────────────────
admin3_proj = phl_admin_3.to_crs(CRS_METRIC)

print(f"Track points loaded : {len(tracks_hist):,}")
print(f"Date range          : {tracks_hist['valid_time'].min().date()} → {tracks_hist['valid_time'].max().date()}")
print(f"Unique storms       : {tracks_hist['storm_id'].nunique()}")

### Option A  -  Fixed Buffer

IBTrACS `gust_speed` has 0% coverage across the historical record, so we cannot apply the 194 km/h gust threshold directly. Instead we derive the equivalent **sustained wind** threshold using a standard tropical cyclone gust factor of **1.3**:

> sustained threshold = 194 ÷ 1.3 ÷ 1.852 ≈ **80.6 knots (149 km/h)**

Each track point at or above this threshold is buffered by 100 km. All circles belonging to the same storm are dissolved into one footprint before the spatial join.

In [ ]:
SUSTAINED_EQUIVALENT_KT = (GUST_THRESHOLD_KMH / GUST_FACTOR) / 1.852  # ~80.6 kt

# Filter to track points at/above the sustained-wind threshold
intense_a = tracks_hist[tracks_hist["wind_speed"] >= SUSTAINED_EQUIVALENT_KT].copy()

# Project, buffer each point, then dissolve to one footprint per storm
intense_a_proj = intense_a.to_crs(CRS_METRIC)
intense_a_proj["geometry"] = intense_a_proj.geometry.buffer(BUFFER_M)
footprints_a = intense_a_proj.dissolve(by="storm_id").reset_index()[["storm_id", "geometry"]]

# Spatial join: which admin-3 municipalities fall inside each storm footprint?
joined_a = gpd.sjoin(admin3_proj, footprints_a, how="inner", predicate="intersects")

# Count unique municipalities per storm
muni_count_a = (
    joined_a.groupby("storm_id")
    .apply(lambda x: x.index.nunique(), include_groups=False)
    .reset_index(name="n_munis")
)

# Attach max wind speed, readable name, and year
muni_count_a = muni_count_a.merge(
    intense_a.groupby("storm_id")["wind_speed"].max().reset_index().rename(columns={"wind_speed": "max_wind_kt"}),
    on="storm_id",
)
muni_count_a["max_wind_kmh"] = (muni_count_a["max_wind_kt"] * 1.852).round(1)
muni_count_a["name"] = muni_count_a["storm_id"].str.split("_").str[0].str.capitalize()
muni_count_a["year"] = muni_count_a["storm_id"].str.extract(r"_wp_(\d{4})$")

exposure_a = (
    muni_count_a[["storm_id", "name", "year", "max_wind_kmh", "n_munis"]]
    .sort_values("n_munis", ascending=False)
    .reset_index(drop=True)
)
exposure_a.index += 1

print(f"Storms with exposure ≥ 1 municipality: {len(exposure_a)}")
exposure_a.drop(columns="storm_id").rename(columns={"n_munis": "municipalities_exposed"})

In [ ]:
# Set the start year  -  applied to exposure_a, exposure_b, and exposure_c
start_year = 2006

exposure_a = exposure_a[exposure_a["year"].astype(float) >= start_year].reset_index(drop=True)
exposure_a.index += 1
print(f"100 km buffer  -  storms from {start_year} onwards: {len(exposure_a)}")

### Option B  -  IBTrACS 64-knot Wind Radii

IBTrACS reports the radius of **64-knot (118 km/h) winds** in four quadrants (NE, SE, SW, NW) in nautical miles, sourced from multiple agencies (e.g. `usa_`, `bom_`, `tokyo_`). Where available, these allow us to build an **asymmetric wind polygon** that reflects the actual shape of the storm's wind field rather than a uniform circle.

Coverage varies by agency and era. To maximise the number of usable track points, all `*radius_64*` columns are discovered automatically and combined: for each point the first non-null value across agencies (ranked by overall coverage) is used. If a quadrant value within that reading is `NaN`, it is filled with the mean of the remaining quadrants. Points where all agencies report all-NaN are skipped. If combined coverage is still too low (< 5%), Option A is used as the primary method.

In [ ]:
def parse_radii(val):
    """
    Parse a PostgreSQL array string like '{130.0,NaN,140.0,NaN}' into a
    list of floats [NE, SE, SW, NW] in nautical miles, or None if all NaN.
    """
    if val is None:
        return None
    parts = str(val).strip("{}").split(",")
    vals = []
    for p in parts:
        try:
            v = float(p.strip())
            vals.append(np.nan if np.isnan(v) else v)
        except ValueError:
            vals.append(np.nan)
    return vals if any(not np.isnan(v) for v in vals) else None


def wind_polygon_utm(x, y, radii_nm):
    """
    Build an asymmetric wind-radius polygon in UTM coordinates.

    Parameters
    ----------
    x, y     : storm centre in metres (UTM)
    radii_nm : [NE, SE, SW, NW] radii in nautical miles
               NaN values are replaced by the mean of valid quadrants.

    Returns
    -------
    shapely Polygon
    """
    nm2m = 1852.0
    valid = [r * nm2m for r in radii_nm if not np.isnan(r)]
    fill  = float(np.mean(valid))
    rm    = [r * nm2m if not np.isnan(r) else fill for r in radii_nm]

    pts = []
    for deg in np.linspace(0, 360, 360, endpoint=False):
        if   deg <  90: r = rm[0]   # NE
        elif deg < 180: r = rm[1]   # SE
        elif deg < 270: r = rm[2]   # SW
        else:           r = rm[3]   # NW
        rad = np.radians(90 - deg)
        pts.append((x + r * np.cos(rad), y + r * np.sin(rad)))

    poly = Polygon(pts)
    return poly if poly.is_valid else poly.buffer(0)


# ── Discover all 64-knot radius columns ──────────────────────────────────────
r64_cols = sorted([c for c in tracks_hist.columns if "radius_64" in c.lower()])
print("64-knot radius columns found:")
for col in r64_cols:
    n_valid = tracks_hist[col].apply(lambda v: parse_radii(v) is not None).sum()
    pct     = n_valid / len(tracks_hist) * 100
    print(f"  {col:40s}: {n_valid:>6,} / {len(tracks_hist):,} points  ({pct:.1f}%)")

# ── Combine: for each row use first non-null value across all columns ─────────
# Sort columns by coverage descending so the best source is tried first
r64_cols_ranked = sorted(
    r64_cols,
    key=lambda c: tracks_hist[c].apply(lambda v: parse_radii(v) is not None).sum(),
    reverse=True,
)

def pick_best_radii(row):
    """Return parsed radii from the first column that has valid data."""
    for col in r64_cols_ranked:
        radii = parse_radii(row[col])
        if radii is not None:
            return radii
    return None

if r64_cols_ranked:
    tracks_hist["_radii_64"] = tracks_hist[r64_cols_ranked].apply(pick_best_radii, axis=1)
    n_combined = tracks_hist["_radii_64"].notna().sum()
    pct_combined = n_combined / len(tracks_hist) * 100
    print(f"\nCombined coverage across all agencies: {n_combined:,} / {len(tracks_hist):,} points  ({pct_combined:.1f}%)")
else:
    tracks_hist["_radii_64"] = None
    pct_combined = 0.0
    print("\nNo 64-knot radius columns found.")

# ── Build Option B exposure table ─────────────────────────────────────────────
if pct_combined >= 5:
    pts_b = gpd.GeoDataFrame(
        tracks_hist[tracks_hist["_radii_64"].notna()].copy(),
        geometry="geometry", crs="EPSG:4326"
    ).to_crs(CRS_METRIC)

    polys = []
    for _, row in pts_b.iterrows():
        radii = row["_radii_64"]
        if radii is None:
            continue
        poly = wind_polygon_utm(row.geometry.x, row.geometry.y, radii)
        if poly is not None:
            polys.append({"storm_id": row["storm_id"], "geometry": poly})

    wind_polys_b = gpd.GeoDataFrame(polys, crs=CRS_METRIC)

    footprints_b = wind_polys_b.dissolve(by="storm_id").reset_index()[["storm_id", "geometry"]]
    joined_b     = gpd.sjoin(admin3_proj, footprints_b, how="inner", predicate="intersects")

    muni_count_b = (
        joined_b.groupby("storm_id")
        .apply(lambda x: x.index.nunique(), include_groups=False)
        .reset_index(name="n_munis")
    )
    muni_count_b = muni_count_b.merge(
        tracks_hist.groupby("storm_id")["wind_speed"].max().reset_index().rename(columns={"wind_speed": "max_wind_kt"}),
        on="storm_id",
    )
    muni_count_b["max_wind_kmh"] = (muni_count_b["max_wind_kt"] * 1.852).round(1)
    muni_count_b["name"] = muni_count_b["storm_id"].str.split("_").str[0].str.capitalize()
    muni_count_b["year"] = muni_count_b["storm_id"].str.extract(r"_wp_(\d{4})$")

    exposure_b = (
        muni_count_b[["storm_id", "name", "year", "max_wind_kmh", "n_munis"]]
        .sort_values("n_munis", ascending=False)
        .reset_index(drop=True)
    )
    exposure_b.index += 1
    print(f"\nOption B  -  storms with exposure ≥ 1 municipality: {len(exposure_b)}")
    display(exposure_b.drop(columns="storm_id").rename(columns={"n_munis": "municipalities_exposed"}))

else:
    exposure_b = None
    print(
        f"\n64-knot radii coverage too low ({pct_combined:.1f}%) for a reliable Option B analysis.\n"
        "Option A (fixed buffer) is the primary method for this record."
    )

### Wind Radii with Mean Infill  -  Primary Method

This is the approach used in all downstream analyses. It uses IBTrACS 64-knot quadrant wind radii where observed (from any agency column) and fills gaps with the **per-quadrant mean** computed across all track points that have valid radii data. This gives complete spatial coverage across the full record without falling back to an arbitrary fixed circle.

All track points with wind speed ≥ 64 kt are included. For each:
- **Radii available**: use the observed asymmetric quadrant values
- **Radii missing**: substitute the dataset-wide per-quadrant mean

In [ ]:
# ── Per-quadrant mean from all track points that have observed radii ──────────
valid_radii = tracks_hist["_radii_64"].dropna().tolist()
mean_radii  = [
    float(np.nanmean([r[i] for r in valid_radii])) for i in range(4)
]
print("Mean 64kt radii used as infill (NE / SE / SW / NW):")
print("  " + " / ".join(f"{r:.0f} nm" for r in mean_radii))

# ── Track points at or above 64 kt ───────────────────────────────────────────
KT_THRESHOLD = 80.0
pts_c = gpd.GeoDataFrame(
    tracks_hist[tracks_hist["wind_speed"] >= KT_THRESHOLD].copy(),
    geometry="geometry", crs="EPSG:4326"
).to_crs(CRS_METRIC)

n_obs    = pts_c["_radii_64"].notna().sum()
n_filled = pts_c["_radii_64"].isna().sum()
print(f"\nTrack points ≥ {KT_THRESHOLD:.0f} kt : {len(pts_c):,}")
print(f"  Observed radii : {n_obs:,} ({n_obs/len(pts_c)*100:.1f}%)")
print(f"  Mean-filled    : {n_filled:,} ({n_filled/len(pts_c)*100:.1f}%)")

# ── Build wind polygons: observed radii where available, mean otherwise ───────
polys_c = []
for _, row in pts_c.iterrows():
    radii = row["_radii_64"] if row["_radii_64"] is not None else mean_radii
    poly  = wind_polygon_utm(row.geometry.x, row.geometry.y, radii)
    if poly is not None:
        polys_c.append({"storm_id": row["storm_id"], "geometry": poly})

wind_polys_c = gpd.GeoDataFrame(polys_c, crs=CRS_METRIC)

# ── Dissolve to one footprint per storm, spatial join to admin-3 ──────────────
footprints_c = wind_polys_c.dissolve(by="storm_id").reset_index()[["storm_id", "geometry"]]
joined_c     = gpd.sjoin(admin3_proj, footprints_c, how="inner", predicate="intersects")

muni_count_c = (
    joined_c.groupby("storm_id")
    .apply(lambda x: x.index.nunique(), include_groups=False)
    .reset_index(name="n_munis")
)
muni_count_c = muni_count_c.merge(
    tracks_hist.groupby("storm_id")["wind_speed"].max().reset_index().rename(columns={"wind_speed": "max_wind_kt"}),
    on="storm_id",
)
muni_count_c["max_wind_kmh"] = (muni_count_c["max_wind_kt"] * 1.852).round(1)
muni_count_c["name"] = muni_count_c["storm_id"].str.split("_").str[0].str.capitalize()
muni_count_c["year"] = muni_count_c["storm_id"].str.extract(r"_wp_(\d{4})$")

exposure_c = (
    muni_count_c[["storm_id", "name", "year", "max_wind_kmh", "n_munis"]]
    .sort_values("n_munis", ascending=False)
    .reset_index(drop=True)
)
exposure_c.index += 1

# Apply the same start_year filter defined in the cell above
exposure_c = exposure_c[exposure_c["year"].astype(float) >= start_year].reset_index(drop=True)
exposure_c.index += 1

print(f"\nStorms with exposure ≥ 1 municipality ({start_year}-2024): {len(exposure_c)}")
display(exposure_c.drop(columns="storm_id").rename(columns={"n_munis": "municipalities_exposed"}))

## Distribution of Municipality Exposure

The plots below compare how municipality exposure varies across the historical record for both approaches:
1. **Histogram**  -  distribution of municipalities exposed per storm
2. **Scatter**  -  relationship between peak wind speed and exposure count

In [ ]:
def plot_exposure_distribution(df, label, color):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor("#f8f8f8")

    ax = axes[0]
    ax.set_facecolor("#f8f8f8")
    ax.hist(df["n_munis"], bins=20, color=color, edgecolor="white", linewidth=0.6)
    ax.axvline(df["n_munis"].median(), color="#333", linestyle="--", linewidth=1.2,
               label=f"Median: {df['n_munis'].median():.0f}")
    ax.axvline(df["n_munis"].mean(), color="#888", linestyle=":", linewidth=1.2,
               label=f"Mean: {df['n_munis'].mean():.0f}")
    ax.set_xlabel("Municipalities exposed", fontsize=11)
    ax.set_ylabel("Number of storms", fontsize=11)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.set_title("Distribution of municipality exposure", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)

    ax = axes[1]
    ax.set_facecolor("#f8f8f8")
    scatter = ax.scatter(
        df["max_wind_kmh"], df["n_munis"],
        c=df["n_munis"], cmap="YlOrRd", s=55, edgecolors="#aaa", linewidths=0.4,
        norm=mcolors.Normalize(vmin=0, vmax=df["n_munis"].max()), zorder=3,
    )
    for _, row in df.nlargest(5, "n_munis").iterrows():
        ax.annotate(
            row["name"], xy=(row["max_wind_kmh"], row["n_munis"]),
            xytext=(4, 3), textcoords="offset points", fontsize=7.5, color="#333",
        )
    fig.colorbar(scatter, ax=ax, shrink=0.7, label="Municipalities exposed")
    ax.set_xlabel("Max wind speed (km/h)", fontsize=11)
    ax.set_ylabel("Municipalities exposed", fontsize=11)
    ax.set_title("Peak wind speed vs municipality exposure", fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(
        f"Historical municipality exposure  -  {len(df)} storms, {start_year}-2024"
        f"\nUsing {label}",
        fontsize=12, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.show()


plot_exposure_distribution(exposure_c, "wind radii (mean-filled)", "#2a9d8f")

In [ ]:
TOP_N = 20

def plot_top_storms(df, label):
    plot_df = df.head(TOP_N).copy()
    plot_df["bar_label"] = plot_df["name"] + " (" + plot_df["year"].fillna("?") + ")"
    norm_w  = mcolors.Normalize(vmin=plot_df["max_wind_kmh"].min(), vmax=plot_df["max_wind_kmh"].max())
    colors  = [cm.YlOrRd(norm_w(v)) for v in plot_df["max_wind_kmh"]]

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor("#f8f8f8")
    ax.set_facecolor("#f8f8f8")

    bars = ax.barh(
        plot_df["bar_label"][::-1], plot_df["n_munis"][::-1],
        color=colors[::-1], edgecolor="white", linewidth=0.6, height=0.7,
    )
    for bar, val in zip(bars, plot_df["n_munis"][::-1]):
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
                str(val), va="center", ha="left", fontsize=8.5, color="#333")

    sm = cm.ScalarMappable(cmap=cm.YlOrRd, norm=norm_w)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02, aspect=20, label="Max wind speed (km/h)")

    ax.set_xlabel("Municipalities exposed", fontsize=11)
    ax.set_title(
        f"Top {TOP_N} storms by municipality exposure\nUsing {label}",
        fontsize=12, fontweight="bold", pad=12,
    )
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.xaxis.grid(True, color="white", linewidth=1.2)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    plt.tight_layout()
    plt.show()


plot_top_storms(exposure_c, "wind radii (mean-filled)")

## Case Study: Typhoon Man-yi (2024)

Man-yi is the second most impactful storm in the record under Option A. The map below shows exposed municipalities highlighted by region (admin level 1 choropleth), with the storm track overlaid.

In [ ]:
manyi_id       = exposure_c[exposure_c["name"].str.lower() == "man-yi"]["storm_id"].values[0]
manyi_muni_idx = joined_c[joined_c["storm_id"] == manyi_id].index.unique()
manyi_munis    = phl_admin_3.loc[manyi_muni_idx]

manyi_track   = tracks_hist[tracks_hist["storm_id"] == manyi_id].sort_values("valid_time")
manyi_intense = manyi_track[manyi_track["wind_speed"] >= KT_THRESHOLD]
manyi_line    = LineString(list(zip(manyi_track.geometry.x, manyi_track.geometry.y)))

munis_proj        = manyi_munis.to_crs(CRS_METRIC)
admin1_proj_local = phl_admin_1.to_crs(CRS_METRIC)
munis_with_rgn    = gpd.sjoin(munis_proj, admin1_proj_local[["geometry"]], how="left", predicate="within")
region_counts  = munis_with_rgn.groupby("index_right").size().reset_index(name="n_munis")
admin1_counts  = phl_admin_1.copy()
admin1_counts  = admin1_counts.merge(region_counts.rename(columns={"index_right": "adm1_idx"}),
                                     left_index=True, right_on="adm1_idx", how="left")
admin1_counts["n_munis"] = admin1_counts["n_munis"].fillna(0).astype(int)
adm1_name_col  = next((c for c in phl_admin_1.columns if "adm1_en" in c.lower()), None)

fig, ax = plt.subplots(figsize=(9, 11))
ax.set_facecolor("#cde0f0")
norm_r = mcolors.Normalize(vmin=0, vmax=admin1_counts["n_munis"].max())
admin1_counts.plot(ax=ax, column="n_munis", cmap="Oranges", norm=norm_r,
                   edgecolor="#888", linewidth=0.8, zorder=2)
manyi_munis.plot(ax=ax, color="none", edgecolor="white", linewidth=0.4, zorder=3)

xs, ys = manyi_line.xy
ax.plot(xs, ys, color="#222", linewidth=1.2, alpha=0.35, linestyle="--", zorder=4)
if len(manyi_intense) >= 2:
    ax.plot(list(manyi_intense.geometry.x), list(manyi_intense.geometry.y),
            color="#1a1a1a", linewidth=2.5, zorder=5,
            label=f"Track ≥ {KT_THRESHOLD:.0f} kt")

step = max(1, len(manyi_track) // 8)
for i in range(0, len(manyi_track) - step, step):
    p0, p1 = manyi_track.iloc[i].geometry, manyi_track.iloc[i + step].geometry
    ax.annotate("", xy=(p1.x, p1.y), xytext=(p0.x, p0.y),
                arrowprops=dict(arrowstyle="-|>", color="#444", lw=0.8, mutation_scale=10), zorder=6)

for _, row in admin1_counts[admin1_counts["n_munis"] > 0].iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    lbl = (f"{row[adm1_name_col]}\n{row['n_munis']}" if adm1_name_col else str(row["n_munis"]))
    ax.text(cx, cy, lbl, ha="center", va="center", fontsize=6.5, fontweight="bold",
            color="white", zorder=7,
            bbox=dict(boxstyle="round,pad=0.2", fc="#333", ec="none", alpha=0.55))

sm = cm.ScalarMappable(cmap="Oranges", norm=norm_r)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=18)
cbar.set_label("Municipalities exposed", fontsize=10)

minx, miny, maxx, maxy = phl_admin_3.total_bounds
ax.set_xlim(minx - 0.8, maxx + 0.8)
ax.set_ylim(miny - 0.8, maxy + 0.8)
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title(
    f"Typhoon Man-yi (2024)  -  {len(manyi_munis)} municipalities exposed\n"
    f"Wind radii (mean-filled), threshold ≥ {KT_THRESHOLD:.0f} kt",
    fontsize=12, fontweight="bold", pad=12,
)
ax.legend(fontsize=9, loc="lower left", framealpha=0.85)
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.show()

## Population Exposure

WorldPop gridded population data (1 km resolution) is used to estimate the number of people living in each municipality. The global raster is first clipped to the Philippines bounding box to keep processing efficient. Zonal statistics then sum the population within each admin-3 polygon. This population layer is joined to the storm exposure results to compute **people exposed** per storm.

In [ ]:
# ── Clip raster to Philippines bounds ────────────────────────────────────────
# Adding a small buffer avoids edge effects at the boundary
minx, miny, maxx, maxy = phl_admin_3.total_bounds
pop_phl = global_pop_raster.rio.clip_box(
    minx=minx - 0.1, miny=miny - 0.1,
    maxx=maxx + 0.1, maxy=maxy + 0.1,
)
print(f"Clipped raster shape : {pop_phl.shape}")
print(f"CRS                  : {pop_phl.rio.crs}")
print(f"NoData value         : {pop_phl.rio.nodata}")

# ── Zonal statistics: population sum per admin-3 polygon ─────────────────────
# Reproject admin-3 to the raster CRS so geometries align correctly
admin3_raster_crs = phl_admin_3.to_crs(pop_phl.rio.crs)

pop_array     = pop_phl.values.squeeze().astype(float)
pop_transform = pop_phl.rio.transform()
nodata_val    = pop_phl.rio.nodata

stats = zonal_stats(
    admin3_raster_crs,
    pop_array,
    affine=pop_transform,
    stats=["sum"],
    nodata=nodata_val,
    all_touched=True,   # include pixels that touch polygon edges
)

# Attach population to admin-3 GeoDataFrame
phl_admin_3_pop = phl_admin_3.copy()
phl_admin_3_pop["population"] = [
    int(round(s["sum"])) if s["sum"] is not None else 0 for s in stats
]

total_pop = phl_admin_3_pop["population"].sum()
print(f"\nTotal population (Philippines): {total_pop:,.0f}")
phl_admin_3_pop[["population"]].describe().round(0)

In [ ]:
pop_lookup = phl_admin_3_pop["population"]

exposure_c["population_exposed"] = exposure_c["storm_id"].apply(
    lambda sid: pop_lookup.loc[joined_c[joined_c["storm_id"] == sid].index.unique()].sum()
)

print("Top 10 storms by people exposed (wind radii, mean-filled):")
display(
    exposure_c[["name", "year", "n_munis", "population_exposed"]]
    .rename(columns={"n_munis": "municipalities_exposed"})
    .head(10)
    .assign(population_exposed=lambda d: d["population_exposed"].apply(lambda v: f"{v:,.0f}"))
)

## Summary Table: Exposure, Combined Input & CERF by Storm

One row per storm joining wind-radii municipality exposure, people exposed, combined input data coverage, and CERF allocation.

In [ ]:
# ── Build join key: lowercase name + year → matches combined_input "typhoon" column
summary = exposure_c[["storm_id", "name", "year", "n_munis", "population_exposed"]].copy()
summary["_key"] = summary["name"].str.lower() + summary["year"].astype(str)

# ── 510 model input: municipalities with perc_dmg ≥ 0.1%% ─────────────────────
cid_agg = (
    combined_input_data[combined_input_data["perc_dmg"] >= 0.1]
    .groupby("typhoon")
    .agg(munis_510=("Mun_Code", "count"))
    .reset_index()
    .rename(columns={"typhoon": "_key"})
)

# ── 510 model input: total houses damaged (housing_units × perc_dmg / 100) ────
houses_agg = (
    combined_input_data
    .assign(houses_damaged=lambda df: df["Housing Units"] * df["perc_dmg"].fillna(0) / 100)
    .groupby("typhoon")
    .agg(houses_damaged_510=("houses_damaged", "sum"))
    .reset_index()
    .rename(columns={"typhoon": "_key"})
)

# ── CERF ─────────────────────────────────────────────────────────────────────
cerf = phl_cerf_data.copy()
cerf["_key_cerf"] = cerf["storm"].str.lower()

# ── Merges ────────────────────────────────────────────────────────────────────
summary = summary.merge(cid_agg,    on="_key", how="left")
summary = summary.merge(houses_agg, on="_key", how="left")
summary = summary.merge(
    cerf[["_key_cerf", "amount"]].rename(columns={"_key_cerf": "_name_lower", "amount": "cerf_usd"}),
    left_on=summary["name"].str.lower(),
    right_on="_name_lower",
    how="left",
).drop(columns=["_key", "_name_lower", "key_0"], errors="ignore")
# Goni's CERF allocation is for 2020; explicitly assign it and clear the 2015 row
_goni_cerf = cerf.loc[cerf["_key_cerf"] == "goni", "amount"]
if not _goni_cerf.empty:
    summary.loc[(summary["name"].str.lower() == "goni") & (summary["year"] == "2020"), "cerf_usd"] = float(_goni_cerf.iloc[0])
summary.loc[(summary["name"].str.lower() == "goni") & (summary["year"] != "2020"), "cerf_usd"] = None

# ── Return periods (Weibull, event-based) ─────────────────────────────────────
N_RP = 2024 - int(exposure_c["year"].dropna().astype(int).min()) + 1

munis_rp = (
    exposure_c[["storm_id", "n_munis"]]
    .sort_values("n_munis", ascending=False)
    .reset_index(drop=True)
)
munis_rp.index += 1
munis_rp["rp_munis"] = (N_RP + 1) / munis_rp.index.values

pop_rp = (
    exposure_c[["storm_id", "population_exposed"]]
    .sort_values("population_exposed", ascending=False)
    .reset_index(drop=True)
)
pop_rp.index += 1
pop_rp["rp_pop"] = (N_RP + 1) / pop_rp.index.values

# ── RP for houses damaged ─────────────────────────────────────────────────────
houses_rp_input = summary[["storm_id", "houses_damaged_510"]].dropna(subset=["houses_damaged_510"])
houses_rp = (
    houses_rp_input
    .sort_values("houses_damaged_510", ascending=False)
    .reset_index(drop=True)
)
houses_rp.index += 1
houses_rp["rp_houses"] = (N_RP + 1) / houses_rp.index.values

summary = summary.merge(munis_rp[["storm_id", "rp_munis"]],   on="storm_id", how="left")
summary = summary.merge(pop_rp[["storm_id",   "rp_pop"]],     on="storm_id", how="left")
summary = summary.merge(houses_rp[["storm_id", "rp_houses"]], on="storm_id", how="left")

# ── Build display table ───────────────────────────────────────────────────────
display_tbl = (
    summary[[
        "name", "year",
        "n_munis", "rp_munis",
        "population_exposed", "rp_pop",
        "houses_damaged_510", "rp_houses", "munis_510", "cerf_usd",
    ]]
    .rename(columns={
        "name":               "Storm",
        "year":               "Year",
        "n_munis":            "Municipalities\nexposed",
        "rp_munis":           "RP\n(muns, yrs)",
        "population_exposed": "People\nexposed",
        "rp_pop":             "RP\n(people, yrs)",
        "munis_510":          "Muns above\n0.1% damage",
        "houses_damaged_510": "510 input\nhouses damaged",
        "rp_houses":          "RP\n(houses, yrs)",
        "cerf_usd":           "CERF\nallocation (USD)",
    })
    .sort_values("People\nexposed", ascending=False)
    .reset_index(drop=True)
)
display_tbl.index += 1

# ── Style ─────────────────────────────────────────────────────────────────────
TEAL            = "#2a9d8f"
HEADER          = "#1f7a70"
HIGHLIGHT_RP    = "#fff3cd"   # warm yellow  -  rows with any RP > 5 years
FOCUS_C      = "#e63946"   # red  -  focus storm rows

FOCUS_STORMS = {"bopha", "durian", "haiyan", "kammuri", "nock-ten", "rai"}

def highlight_row(row):
    storm = str(row.get("Storm", "")).strip().lower()

    if storm in FOCUS_STORMS:
        n = len(row)
        styles = []
        for i in range(n):
            s = f"box-shadow: inset 0 2px 0 0 {FOCUS_C}, inset 0 -2px 0 0 {FOCUS_C}"
            if i == 0:
                s += f"; border-left: 2px solid {FOCUS_C}"
            if i == n - 1:
                s += f"; border-right: 2px solid {FOCUS_C}"
            styles.append(s)
        return styles

    for col in ["RP\n(muns, yrs)", "RP\n(people, yrs)", "RP\n(houses, yrs)"]:
        v = row.get(col)
        if pd.notna(v) and float(v) > 5:
            return [f"background-color: {HIGHLIGHT_RP}" for _ in row]

    return ["" for _ in row]

styled = (
    display_tbl.style
    .format({
        "Municipalities\nexposed":   "{:,.0f}",
        "RP\n(muns, yrs)":           "{:.1f}",
        "People\nexposed":           lambda v: f"{v/1e6:.2f}M" if pd.notna(v) else "",
        "RP\n(people, yrs)":         "{:.1f}",
        "Muns above\n0.1% damage":  lambda v: f"{v:.0f}" if pd.notna(v) else "",
        "510 input\nhouses damaged": lambda v: f"{v:,.0f}" if pd.notna(v) and v > 0 else "",
        "RP\n(houses, yrs)":         lambda v: f"{v:.1f}" if pd.notna(v) else "",
        "CERF\nallocation (USD)":    lambda v: f"${v:,.0f}" if pd.notna(v) else "",
    })
    .set_table_styles([
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("width", "100%%"),
            ("font-family", "Arial, sans-serif"),
            ("font-size", "10px"),
        ]},
        {"selector": "th", "props": [
            ("background-color", HEADER),
            ("color", "white"),
            ("font-size", "10px"),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "5px 8px"),
            ("white-space", "pre-line"),
            ("border", f"1px solid {HEADER}"),
            ("line-height", "1.2"),
        ]},
        {"selector": "td", "props": [
            ("font-size", "10px"),
            ("text-align", "center"),
            ("padding", "3px 8px"),
            ("border-bottom", "1px solid #e8e8e8"),
            ("background-color", "white"),
            ("line-height", "1.2"),
        ]},
        {"selector": "tr:nth-child(even) td", "props": [
            ("background-color", "#f7fdfc"),
        ]},
        {"selector": "th.row_heading, td.row_heading", "props": [
            ("font-weight", "bold"),
            ("border-right", f"2px solid {TEAL}"),
            ("background-color", "white"),
            ("padding", "3px 6px"),
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"),
            ("font-size", "11px"),
            ("font-weight", "bold"),
            ("color", "#333"),
            ("padding-bottom", "6px"),
            ("text-align", "left"),
        ]},
    ])
    .apply(highlight_row, axis=1)
    .set_caption(
        f"Storm exposure summary  -  wind radii (mean-filled), {start_year}-2024   |   "
        f"RP = Weibull event-based return period over {N_RP}-year record "
    )
)

display(styled)


In [ ]:
plot_df = summary[summary[["rp_munis", "rp_pop", "rp_houses"]].notna().all(axis=1)].copy()
plot_df["label"] = plot_df["name"] + " (" + plot_df["year"].fillna("?") + ")"

SIZE_SCALE = 120
RP_THRESHOLD = 3
COLOR_HIGH = "#c2622d"   # burnt orange  -  houses RP >= 3
COLOR_LOW  = "#f5e6c8"   # pale straw  -  houses RP < 3

bubble_colors = plot_df["rp_houses"].apply(
    lambda v: COLOR_HIGH if v >= RP_THRESHOLD else COLOR_LOW
)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor("#f8f8f8")
ax.set_facecolor("#f8f8f8")

# Plot bubbles first so axis limits are set before threshold line labels
ax.scatter(
    plot_df["rp_munis"], plot_df["rp_pop"],
    s=plot_df["rp_houses"] * SIZE_SCALE,
    c=bubble_colors,
    edgecolors="#888", linewidths=0.5,
    alpha=0.88, zorder=3,
)

# Reference lines at RP = 3
ax.axvline(RP_THRESHOLD, color="#999", linewidth=1.1, linestyle="--", zorder=2)
ax.axhline(RP_THRESHOLD, color="#999", linewidth=1.1, linestyle="--", zorder=2)
ax.text(RP_THRESHOLD + 0.05, 0.01, f"RP={RP_THRESHOLD} yr",
        transform=ax.get_xaxis_transform(), fontsize=7.5, color="#666",
        va="bottom", ha="left")
ax.text(0.01, RP_THRESHOLD + 0.05, f"RP={RP_THRESHOLD} yr",
        transform=ax.get_yaxis_transform(), fontsize=7.5, color="#666",
        va="bottom", ha="left")

# Label storms where rp_munis > 3 OR rp_pop > 3
to_label = plot_df[(plot_df["rp_munis"] > RP_THRESHOLD) | (plot_df["rp_pop"] > RP_THRESHOLD)]
for _, row in to_label.iterrows():
    ax.annotate(
        row["label"],
        xy=(row["rp_munis"], row["rp_pop"]),
        xytext=(8, 5), textcoords="offset points",
        fontsize=8, color="#333",
        arrowprops=dict(arrowstyle="-", color="#bbb", lw=0.6),
    )

# Bubble size legend
rp_ref_vals = [v for v in [2, 5, 10, 15] if v <= plot_df["rp_houses"].max() * 1.05]
handles = [
    plt.scatter([], [], s=v * SIZE_SCALE, c="#ddd", edgecolors="#888",
                linewidths=0.5, alpha=0.88, label=f"Houses RP = {v} yr")
    for v in rp_ref_vals
]
handles += [
    plt.scatter([], [], s=60, c=COLOR_HIGH, edgecolors="#888", linewidths=0.5,
                alpha=0.88, label=f"Houses RP ≥ {RP_THRESHOLD} yr"),
    plt.scatter([], [], s=60, c=COLOR_LOW,  edgecolors="#888", linewidths=0.5,
                alpha=0.88, label=f"Houses RP < {RP_THRESHOLD} yr"),
    plt.Line2D([0], [0], color="#999", linewidth=1.1, linestyle="--",
               label=f"RP = {RP_THRESHOLD} yr threshold"),
]
ax.legend(handles=handles, fontsize=8, loc="upper left",
          framealpha=0.85, edgecolor="#ddd", title="Legend", title_fontsize=8)

ax.set_xlabel("Return period  -  municipalities exposed (yrs)", fontsize=11)
ax.set_ylabel("Return period  -  people exposed (yrs)", fontsize=11)
ax.set_title(
    "Storm severity  -  return periods by exposure and damage\n"
    "Bubble size = RP of houses damaged  |  Orange: houses RP ≥ 3 yr  |  Dashed: RP = 3 yr",
    fontsize=12, fontweight="bold", pad=12,
)
ax.xaxis.grid(True, color="#e0e0e0", linewidth=0.8)
ax.yaxis.grid(True, color="#e0e0e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.show()


## Correlation: Exposure vs Damage

Pearson correlations between municipality exposure, people exposed, and 510-model houses damaged, across all storms with complete data. Scatter plots show each pair with a fitted regression line and r value.

In [ ]:
from scipy import stats

corr_df = summary[["n_munis", "population_exposed", "houses_damaged_510"]].dropna().copy()
corr_df.columns = ["Muns exposed", "People exposed", "Houses damaged"]

pairs = [
    ("Muns exposed",   "People exposed"),
    ("Muns exposed",   "Houses damaged"),
    ("People exposed", "Houses damaged"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.patch.set_facecolor("#f8f8f8")

for ax, (xcol, ycol) in zip(axes, pairs):
    ax.set_facecolor("#f8f8f8")
    x = corr_df[xcol]
    y = corr_df[ycol]

    r, p = stats.pearsonr(x, y)

    ax.scatter(x, y, color="#c2622d", edgecolors="#888", linewidths=0.5,
               alpha=0.75, s=45, zorder=3)

    # Regression line
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color="#555", linewidth=1.2, zorder=2)

    p_str = "< 0.001" if p < 0.001 else f"= {p:.3f}"
    ax.text(0.05, 0.93, f"r = {r:.2f} p {p_str}",
            transform=ax.transAxes, fontsize=9, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#ddd", alpha=0.85))

    # Format large axis numbers
    if y.max() > 1e5:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}k"))
    if x.max() > 1e5:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}k"))

    ax.set_xlabel(xcol, fontsize=10)
    ax.set_ylabel(ycol, fontsize=10)
    ax.xaxis.grid(True, color="#e0e0e0", linewidth=0.8)
    ax.yaxis.grid(True, color="#e0e0e0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)

fig.suptitle(
    f"Pairwise correlations across {len(corr_df)} storms with complete data",
    fontsize=12, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

# Print correlation matrix
print("Pearson correlation matrix:")
display(corr_df.corr().round(2).style
    .background_gradient(cmap="RdYlGn", vmin=-1, vmax=1)
    .format("{:.2f}")
)


## Return Period Analysis

Return periods use the **Weibull plotting position**: RP = (N + 1) / rank, where N is the number of years in the record and rank is the storm's position sorted from worst to least severe.

This is an **event-based** (per-storm) return period, not an annual maximum approach. Each storm is one entry regardless of how many storms occurred that year. The RP therefore answers: *on average, how many years between storms at or above this severity level?* If multiple storms hit in one year, all are counted, so the RP can be less than 1 for the very most common events.

In [ ]:
N_YEARS = 2024 - int(exposure_c["year"].dropna().astype(int).min()) + 1
print(f"Record length : {N_YEARS} years ({start_year}-2024)")
print(f"Storms ranked : {len(exposure_c)}  (event-based, not annual maxima)")

def compute_return_period(series, n_years):
    s = series.sort_values(ascending=False).reset_index(drop=True)
    s.index += 1
    rp = pd.Series((n_years + 1) / s.index.values, index=s.index)
    return s, rp

rp_munis, rp_munis_yrs = compute_return_period(exposure_c["n_munis"],           N_YEARS)
rp_pop,   rp_pop_yrs   = compute_return_period(exposure_c["population_exposed"], N_YEARS)

# Build label lookup: value → "Name (year)"   -  top 10 per metric
def top_labels(df, col, n=10):
    top = df.nlargest(n, col)
    return dict(zip(top[col], top["name"] + " (" + top["year"].fillna("?") + ")"))

munis_labels = top_labels(exposure_c, "n_munis")
pop_labels   = top_labels(exposure_c, "population_exposed")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor("#f8f8f8")

for ax, vals, rp_yrs, labels, ylabel, color in [
    (axes[0], rp_munis, rp_munis_yrs, munis_labels, "Municipalities exposed", "#2a9d8f"),
    (axes[1], rp_pop,   rp_pop_yrs,   pop_labels,   "People exposed",         "#2a9d8f"),
]:
    ax.set_facecolor("#f8f8f8")
    ax.plot(rp_yrs.values, vals.values, "o-", color=color, linewidth=1.8,
            markersize=5, markerfacecolor="white", markeredgewidth=1.5, zorder=3)

    for val, label in labels.items():
        idx  = (vals - val).abs().idxmin()
        rp_v = rp_yrs.loc[idx]
        ax.annotate(
            label,
            xy=(rp_v, val),
            xytext=(7, 0), textcoords="offset points",
            fontsize=7, color="#333", va="center",
            arrowprops=dict(arrowstyle="-", color="#ccc", lw=0.5),
        )

    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:g}"))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    ax.set_xlabel("Return period (years)", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(ylabel, fontsize=11, fontweight="bold")
    ax.xaxis.grid(True, which="both", color="#ddd", linewidth=0.8)
    ax.yaxis.grid(True, color="#ddd", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    for rp_ref in [2, 3, 5, 10]:
        if rp_ref <= N_YEARS:
            ax.axvline(rp_ref, color="#aaa", linewidth=0.8, linestyle="--")
            ax.text(rp_ref * 1.05, ax.get_ylim()[1] * 0.97,
                    f"{rp_ref}yr", fontsize=7, color="#888", va="top")

plt.suptitle(
    f"Event-based return period - wind radii (mean-filled)\n"
    f"{N_YEARS}-year record, {start_year}-2024  ·  RP = (N+1) / rank per storm",
    fontsize=11, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor("#f8f8f8")
ax.set_facecolor("#f8f8f8")

norm_w = mcolors.Normalize(
    vmin=exposure_c["max_wind_kmh"].min(),
    vmax=exposure_c["max_wind_kmh"].max(),
)
scatter = ax.scatter(
    exposure_c["n_munis"],
    exposure_c["population_exposed"],
    c=exposure_c["max_wind_kmh"], cmap="YlOrRd", norm=norm_w,
    s=70, edgecolors="#aaa", linewidths=0.5, zorder=3,
)
for _, row in exposure_c.nlargest(8, "population_exposed").iterrows():
    ax.annotate(
        f"{row['name']} ({row['year']})",
        xy=(row["n_munis"], row["population_exposed"]),
        xytext=(5, 3), textcoords="offset points",
        fontsize=8, color="#333",
    )
fig.colorbar(scatter, ax=ax, label="Max wind speed (km/h)", shrink=0.8)
ax.set_xlabel("Municipalities exposed", fontsize=11)
ax.set_ylabel("People exposed (millions)", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.set_title(
    "Municipalities vs people exposed per storm\nUsing wind radii (mean-filled), colored by max wind speed",
    fontsize=12, fontweight="bold", pad=12,
)
ax.xaxis.grid(True, color="#ddd", linewidth=0.8)
ax.yaxis.grid(True, color="#ddd", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
noru_row       = exposure_c[exposure_c["name"].str.lower() == "noru"].iloc[0]
noru_id        = noru_row["storm_id"]
noru_muni_idx  = joined_c[joined_c["storm_id"] == noru_id].index.unique()
noru_munis     = phl_admin_3_pop.loc[noru_muni_idx]
noru_pop       = noru_munis["population"].sum()

noru_track   = tracks_hist[tracks_hist["storm_id"] == noru_id].sort_values("valid_time")
noru_intense = noru_track[noru_track["wind_speed"] >= KT_THRESHOLD]
noru_line    = LineString(list(zip(noru_track.geometry.x, noru_track.geometry.y)))

munis_proj        = noru_munis.to_crs(CRS_METRIC)
admin1_proj_local = phl_admin_1.to_crs(CRS_METRIC)
munis_with_rgn    = gpd.sjoin(munis_proj, admin1_proj_local[["geometry"]], how="left", predicate="within")
region_counts     = munis_with_rgn.groupby("index_right").size().reset_index(name="n_munis")
admin1_counts     = phl_admin_1.copy()
admin1_counts     = admin1_counts.merge(
    region_counts.rename(columns={"index_right": "adm1_idx"}),
    left_index=True, right_on="adm1_idx", how="left",
)
admin1_counts["n_munis"] = admin1_counts["n_munis"].fillna(0).astype(int)
adm1_name_col = next((c for c in phl_admin_1.columns if "adm1_en" in c.lower()), None)

fig, ax = plt.subplots(figsize=(9, 11))
ax.set_facecolor("#cde0f0")
norm_r = mcolors.Normalize(vmin=0, vmax=admin1_counts["n_munis"].max())
admin1_counts.plot(ax=ax, column="n_munis", cmap="Oranges", norm=norm_r,
                   edgecolor="#888", linewidth=0.8, zorder=2)
noru_munis.plot(ax=ax, color="none", edgecolor="white", linewidth=0.4, zorder=3)

xs, ys = noru_line.xy
ax.plot(xs, ys, color="#222", linewidth=1.2, alpha=0.35, linestyle="--", zorder=4)
if len(noru_intense) >= 2:
    ax.plot(list(noru_intense.geometry.x), list(noru_intense.geometry.y),
            color="#1a1a1a", linewidth=2.5, zorder=5,
            label=f"Track ≥ {KT_THRESHOLD:.0f} kt")

step = max(1, len(noru_track) // 8)
for i in range(0, len(noru_track) - step, step):
    p0, p1 = noru_track.iloc[i].geometry, noru_track.iloc[i + step].geometry
    ax.annotate("", xy=(p1.x, p1.y), xytext=(p0.x, p0.y),
                arrowprops=dict(arrowstyle="-|>", color="#444", lw=0.8, mutation_scale=10), zorder=6)

for _, row in admin1_counts[admin1_counts["n_munis"] > 0].iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    lbl = (f"{row[adm1_name_col]}\n{row['n_munis']}" if adm1_name_col else str(row["n_munis"]))
    ax.text(cx, cy, lbl, ha="center", va="center", fontsize=6.5, fontweight="bold",
            color="white", zorder=7,
            bbox=dict(boxstyle="round,pad=0.2", fc="#333", ec="none", alpha=0.55))

sm = cm.ScalarMappable(cmap="Oranges", norm=norm_r)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=18)
cbar.set_label("Municipalities exposed", fontsize=10)

minx, miny, maxx, maxy = phl_admin_3.total_bounds
ax.set_xlim(minx - 0.8, maxx + 0.8)
ax.set_ylim(miny - 0.8, maxy + 0.8)
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title(
    f"Typhoon Noru ({noru_row['year']})  -  {len(noru_munis)} municipalities exposed\n"
    f"People exposed: {noru_pop:,.0f}  |  Wind radii (mean-filled), threshold ≥ {KT_THRESHOLD:.0f} kt",
    fontsize=12, fontweight="bold", pad=12,
)
ax.legend(fontsize=9, loc="lower left", framealpha=0.85)
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.show()

## Focus Storm Maps  -  Track & Wind Radii

One map per focus storm showing:
- Philippine admin-1 choropleth coloured by number of exposed municipalities
- Individual 64-kt wind radius polygons (blue, semi-transparent) around each intense track point  -  the spatial footprint used to determine municipality exposure
- Full storm track (dashed) and the intense segment (≥ 80 kt, solid) with direction arrows


In [ ]:
def plot_storm_case_study_with_radii(storm_name_lower):
    mask = exposure_c["name"].str.lower() == storm_name_lower
    if not mask.any():
        print(f"Storm '{storm_name_lower}' not found in exposure_c")
        return

    row_exp  = exposure_c[mask].iloc[0]
    storm_id = row_exp["storm_id"]

    muni_idx = joined_c[joined_c["storm_id"] == storm_id].index.unique()
    munis    = phl_admin_3.loc[muni_idx]

    track       = tracks_hist[tracks_hist["storm_id"] == storm_id].sort_values("valid_time")
    intense_pts = track[track["wind_speed"] >= KT_THRESHOLD]
    track_line  = (
        LineString(list(zip(track.geometry.x, track.geometry.y)))
        if len(track) >= 2 else None
    )

    # Individual wind-radius polygons for this storm (before dissolve)
    storm_radii = wind_polys_c[wind_polys_c["storm_id"] == storm_id]

    # Admin-1 choropleth by region
    munis_proj        = munis.to_crs(CRS_METRIC)
    admin1_proj_local = phl_admin_1.to_crs(CRS_METRIC)
    munis_with_rgn    = gpd.sjoin(munis_proj, admin1_proj_local[["geometry"]],
                                  how="left", predicate="within")
    region_counts  = munis_with_rgn.groupby("index_right").size().reset_index(name="n_munis")
    admin1_counts  = phl_admin_1.copy()
    admin1_counts  = admin1_counts.merge(
        region_counts.rename(columns={"index_right": "adm1_idx"}),
        left_index=True, right_on="adm1_idx", how="left",
    )
    admin1_counts["n_munis"] = admin1_counts["n_munis"].fillna(0).astype(int)
    adm1_name_col = next((c for c in phl_admin_1.columns if "adm1_en" in c.lower()), None)

    fig, ax = plt.subplots(figsize=(9, 11))
    ax.set_facecolor("#cde0f0")

    norm_r = mcolors.Normalize(vmin=0, vmax=max(admin1_counts["n_munis"].max(), 1))
    admin1_counts.plot(ax=ax, column="n_munis", cmap="Oranges", norm=norm_r,
                       edgecolor="#888", linewidth=0.8, zorder=2)
    munis.plot(ax=ax, color="none", edgecolor="white", linewidth=0.4, zorder=3)

    # Wind radii circles (individual polygons per intense track point)
    if len(storm_radii) > 0:
        storm_radii_geo = storm_radii.to_crs("EPSG:4326")
        storm_radii_geo.plot(
            ax=ax, color="#5b9bd5", alpha=0.22,
            edgecolor="#2c5f8a", linewidth=0.7, zorder=4,
            label="Wind radii (64 kt)",
        )

    # Track
    if track_line is not None:
        xs, ys = track_line.xy
        ax.plot(xs, ys, color="#222", linewidth=1.2, alpha=0.35, linestyle="--", zorder=5)

    if len(intense_pts) >= 2:
        ax.plot(
            list(intense_pts.geometry.x), list(intense_pts.geometry.y),
            color="#1a1a1a", linewidth=2.5, zorder=6,
            label=f"Track ≥ {KT_THRESHOLD:.0f} kt",
        )

    # Direction arrows
    step = max(1, len(track) // 8)
    for i in range(0, len(track) - step, step):
        p0, p1 = track.iloc[i].geometry, track.iloc[i + step].geometry
        ax.annotate(
            "", xy=(p1.x, p1.y), xytext=(p0.x, p0.y),
            arrowprops=dict(arrowstyle="-|>", color="#444", lw=0.8, mutation_scale=10),
            zorder=7,
        )

    # Region labels
    for _, row in admin1_counts[admin1_counts["n_munis"] > 0].iterrows():
        cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
        lbl = (f"{row[adm1_name_col]}\n{row['n_munis']}" if adm1_name_col
               else str(row["n_munis"]))
        ax.text(cx, cy, lbl, ha="center", va="center", fontsize=6.5, fontweight="bold",
                color="white", zorder=8,
                bbox=dict(boxstyle="round,pad=0.2", fc="#333", ec="none", alpha=0.55))

    sm = cm.ScalarMappable(cmap="Oranges", norm=norm_r)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=18)
    cbar.set_label("Municipalities exposed", fontsize=10)

    minx, miny, maxx, maxy = phl_admin_3.total_bounds
    ax.set_xlim(minx - 0.8, maxx + 0.8)
    ax.set_ylim(miny - 0.8, maxy + 0.8)
    ax.set_xlabel("Longitude", fontsize=11)
    ax.set_ylabel("Latitude", fontsize=11)

    display_name = storm_name_lower.title()
    ax.set_title(
        f"Typhoon {display_name} ({row_exp['year']})  -  {len(munis)} municipalities exposed\n"
        f"Blue circles = 64-kt wind radii  |  threshold ≥ {KT_THRESHOLD:.0f} kt",
        fontsize=12, fontweight="bold", pad=12,
    )
    ax.legend(fontsize=9, loc="lower left", framealpha=0.85)
    ax.tick_params(labelsize=9)
    plt.tight_layout()
    plt.show()


for _storm in ["bopha", "durian", "haiyan", "kammuri", "man-yi", "nock-ten", "rai"]:
    plot_storm_case_study_with_radii(_storm)
